In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
import torch

from config import RUNS_PATH
from training.data import make_dataloaders
from training.io import load_or_prepare_encoded
from training.models import DecisionRegressor
from training.trainer import Trainer
from training.training_config_schema import training_config_from_dict
from utils import get_logger, set_random_seeds

In [ ]:
# Choose a run directory to evaluate
run_dir = RUNS_PATH / "20250818-022735"

# Load the config
with open(run_dir / "config_snapshot.json", "r", encoding="utf-8") as f:
    cfg_dict = json.load(f)
cfg = training_config_from_dict(cfg_dict)

# Load the training metrics
train_metrics = pd.read_csv(run_dir / "metrics.csv")

In [ ]:
# Set random seeds for training
set_random_seeds(cfg.seed)

# Prepare the logger
logger = get_logger("train", cfg.log_level, cfg.use_logging)

In [ ]:
# Data
df_train, df_eval, df_test, vocab_sizes = load_or_prepare_encoded(
    force_rebuild=False,
    seed=cfg.seed,
    logger=logger,
)

In [ ]:
_, _, test_dl = make_dataloaders(
    df_train=df_train,
    df_eval=df_eval,
    df_test=df_test,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
    seed=cfg.seed,
    logger=logger,
)

In [ ]:
# Instantiate model
model = DecisionRegressor(
    vocab_sizes=vocab_sizes,
    embed_dim_single=cfg.embed_dim_single,
    embed_dim_multi=cfg.embed_dim_multi,
    hidden=cfg.hidden,
    dropout=cfg.dropout,
)

model_checkpoint_path = run_dir / "best.ckpt"

model.load_state_dict(torch.load(model_checkpoint_path))
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
# Trainer
trainer = Trainer(
    model,
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
    patience=cfg.patience,
    device=cfg.device,
    use_huber=cfg.use_huber,
    logger=logger,
    run_dir=run_dir,
    save_best=False,
)

In [ ]:
# Evaluate
avg_loss, rmse, mae = trainer.evaluate(test_dl)
print("Test metrics:")
print("avg_loss: ", avg_loss)
print("rmse: ", rmse)
print("mae: ", mae)

In [ ]:
# Show the training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Train vs Eval Loss (overfitting check)
axes[0].plot(train_metrics["epoch"], train_metrics["train_loss"], label="Train loss", c="b")
axes[0].plot(train_metrics["epoch"], train_metrics["val_loss"], label="Eval loss", c="r")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss over Epochs (log scale)")
#axes[0].set_yscale("log")   # optional log scale
axes[0].legend()
axes[0].grid(True, which="both")

# 2. Train vs Eval RMSE
axes[1].plot(train_metrics["epoch"], train_metrics["train_rmse"], label="Train RMSE", c="b")
axes[1].plot(train_metrics["epoch"], train_metrics["val_rmse"], label="Eval RMSE", c="r")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("RMSE")
axes[1].set_title("RMSE over Epochs")
axes[1].legend()
axes[1].grid(True)

# 3. Train vs Eval MAE
axes[2].plot(train_metrics["epoch"], train_metrics["train_mae"], label="Train MAE", c="b")
axes[2].plot(train_metrics["epoch"], train_metrics["val_mae"], label="Eval MAE", c="r")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("MAE")
axes[2].set_title("MAE over Epochs")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()
